# Cross-product against relatedness — 1kg_eur

**The headline figure.** Mean phenotype cross-product `E[y_i y_j]` against GRM
relatedness `a_ij`, binned, with the additive prediction drawn through it. Under
purely additive genetics the points lie on a straight line through the origin,
with slope `h2`, all the way from unrelated pairs to full sibs. Any systematic
elevation at `a_ij ~ 0.5` above the line fitted to unrelated pairs is the
"high in sibs" pattern — the result this run exists to measure.

This used to live as "Section A" of `10_crossproduct_diagnostics.ipynb`. It is
not a diagnostic, so it has its own notebook.

Three things are shown that a single curve cannot be:

- **Per class set.** `noPO` (`other` + `FS`, parent-offspring removed) is the
  primary curve; `pooled` is what an unclassified run gives; `PO` is the
  contrast. PO and FS pairs sit at the same `a_ij` but share different things,
  so keeping them apart is the point of the classification.
- **With the ladder's fits overlaid**, so the reader can see what each model in
  `09_estimators.ipynb` is actually claiming about the same points.
- **Zoomed on the unrelated region**, where nearly all the pairs are. The full
  range is dominated visually by a handful of sparse related bins; `a_ij < 0.1`
  holds the bins that determine `h2_Unrel`, and their error bars are invisibly
  small at full scale.

Bin means come from the merge's per-class sums; error bars are **delete-block
jackknife SEs of each bin mean**, recomputed from the `.jk.tsv` replicates the
same way `src/he_estimators.py` does, so a class *set* like `noPO` gets a
correct SE rather than a combination of per-class ones.

**Prerequisites:** `08_accumulate.ipynb` — `*_merged.full.tsv` and
`*_merged.jk.tsv`, plus the bins file they were merged with.

## Setup

In [ ]:
import glob
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

REPO_DIR = os.environ.get("AOUCOV_REPO") or next(
    (p for p in (os.path.expanduser("~/repos/aou-covariance"),
                 os.path.expanduser("~/repos/AOU-covariance"),
                 os.path.expanduser("~/aou-covariance"))
     if os.path.isdir(os.path.join(p, "src"))),
    None,
)
assert REPO_DIR, "cannot find the repo checkout — set AOUCOV_REPO"

WORK_DIR  = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")
BINS_FILE = f"{WORK_DIR}/bins_wide.txt"      # the bins the merge used
NBLOCKS   = 50                               # must match 08_accumulate.ipynb

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
OUT_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}/07_estimators/crossproduct_curves"
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, f"{REPO_DIR}/src")
import importlib
import he_estimators as HE
importlib.reload(HE)

assert os.path.isfile(BINS_FILE), f"missing {BINS_FILE} — see 08_accumulate.ipynb"
MID = HE.load_grid(BINS_FILE)

# Ranges the estimators use, drawn on every panel so the figure and the
# estimator table cannot drift apart.
U_HI, T_HI, R_HI = HE.U_HI, HE.T_HI, HE.R_HI
FS_RANGE = (0.4, 0.6)
# The zoom is bounded on both sides: the lowest bins are a thin, noisy tail
# that stretched the axis for a handful of pairs.
ZOOM_MIN, ZOOM_MAX = -0.03, 0.1
# A floor on how low the zoom's y axis may go, not a fixed limit: one stray
# bin should not be able to stretch the panel, but where the data are tighter
# than this the axis tightens with them.
ZOOM_YFLOOR = -0.15

# Duplicate/MZ pairs sit at a_ij ~ 1 and are not part of this result -- no
# estimator uses them, and on a shared axis they stretch the x-range for a
# handful of pairs. Everything is drawn over a_ij <= MAX_A.
MAX_A = R_HI

# A bin whose mean rests on a few pairs is mostly noise and reads as structure.
# Bins below this pair count are dropped from every curve; the count of dropped
# bins is reported so the threshold is never silently doing a lot of work.
MIN_PAIRS_PER_BIN = 50

# PO and FS occupy the same bins, so PO is nudged along x to keep both readable.
PO_XOFF = 0.004

print(f"bins: {len(MID)} | U<{U_HI} T<{T_HI} R<{R_HI} | FS {FS_RANGE}")
print(f"out : {OUT_DIR}")

## Style

Follows the palette established in `archive/analyses/model_ladder/ladder_cells.md`:
colour does one job per figure, covariate set is an ordered blue ramp, pair
class is a categorical hue.

In [ ]:
FIG_TRANSPARENT = False   # white background; True for transparent
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#ffffff", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")

# Colour is reserved for pair class, and nothing else on these axes carries a
# hue. That is not a stylistic choice: the three class hues below are the first
# three slots of the validated categorical palette and clear the all-pairs
# colourblind floors (worst pair CVD dE 9.2, normal-vision 24.0). Adding a
# fourth hue for a model line does not -- every five-hue combination tried fails
# it, and the worst pairs (red/orange dE 7.1) are hard to separate even with
# full colour vision. So the fitted models are ink, separated by dash pattern
# and named by a direct label at the end of each line.
CLASS_COL = {"other": "#2a78d6", "FS": "#eb6834", "PO": "#1baf7a"}
CLASS_MARK = {"other": "o", "FS": "s", "PO": "^"}
CLASS_LABEL = {"other": "unrelated / distant", "FS": "full sibs", "PO": "parent-offspring"}

# ordered -> sequential ramp, not a categorical set (used for covariate sets)
# The dark end stops at #215596 rather than #104281: the darkest step was
# close enough to the black reference line that a point sitting on the line
# was hard to pick out. Still passes the ordinal checks -- monotone
# lightness, adjacent dL >= 0.06, light end clear of the surface.
RAMP = ["#86b6ef", "#4b93e8", "#2f76cf", "#215596"]

# the four ladder rungs: ink, told apart by dash pattern and a direct label
# MetBrewer "Egypt" -- one of its native 4-colour palettes -- with the yellow
# darkened from #FAB255 to #E89C33. Egypt as published fails the lightness band
# on a light surface (L 0.815, the band tops out at 0.77); snapping that one
# step into the band is the documented remedy and leaves the palette's
# character intact. Validated: worst adjacent CVD dE 8.8, normal-vision 19.0.
# Aqua and yellow are still under 3:1 against the surface, so the relief rule
# applies and every rung keeps a dash pattern and a legend entry.
# Egypt's #0F7BA2 is a teal-blue; it reads apart from the covariate-set ramp
# (#86b6ef..#104281) but that is the one pair to check if the ramp changes.
MODEL_COL = {
    "h2_OneSlope": "#DD5129",
    "h2_Rel": "#0F7BA2",
    "h2_OneSlopeOffsets": "#43B284",
    "h2_RelOffsets": "#E89C33",
}
# Keys are HE.MODELS keys -- the REPORTED models. HE.LADDER is a different set
# (it swaps h2_Rel for a banded twin so the nested comparisons share bins), and
# drawing from it was a bug: LADDER has no "h2_Rel" entry.
FIT_STYLE = {
    "h2_OneSlope":        dict(color=MODEL_COL["h2_OneSlope"],        ls=(0, (1.2, 1.4)),              label="one slope, all pairs"),
    "h2_Rel":             dict(color=MODEL_COL["h2_Rel"],             ls=(0, (5.5, 1.8)),              label="related: slope + intercept"),
    "h2_OneSlopeOffsets": dict(color=MODEL_COL["h2_OneSlopeOffsets"], ls=(0, (3.5, 1.3, 1.1, 1.3)),    label="one slope + band intercepts"),
    "h2_RelOffsets":      dict(color=MODEL_COL["h2_RelOffsets"],      ls=(0, (6, 1.4, 1.1, 1.4, 1.1, 1.4)), label="band slopes + intercepts"),
}
assert set(FIT_STYLE) <= set(HE.MODELS), "FIT_STYLE names a model the module does not define"

# Helvetica where it exists; on Linux this resolves to Liberation/Nimbus Sans,
# which are metric-compatible, and falls back to DejaVu Sans, which is always
# present. mathtext is set to the same family so E[y~ y~] matches the body text.
FONT_STACK = ["Helvetica Neue", "Helvetica", "Liberation Sans", "Nimbus Sans",
              "Arial", "DejaVu Sans"]
from matplotlib import font_manager as _fm
_have = {f.name for f in _fm.fontManager.ttflist}
FONT = next((f for f in FONT_STACK if f in _have), "DejaVu Sans")

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "figure.dpi": 120,
    # set FIG_TRANSPARENT = True above for a transparent background instead of
    # white -- useful on slides, wrong for anything printed on paper
    "savefig.transparent": FIG_TRANSPARENT,
    "font.family": FONT, "font.size": 9,
    "mathtext.fontset": "custom", "mathtext.rm": FONT,
    "mathtext.it": f"{FONT}:italic", "mathtext.bf": f"{FONT}:bold",
    "axes.titlesize": 10, "axes.labelsize": 9,
    # theme_bw / theme_linedraw style: a full panel border and a standard grid
    # on both axes (major plus a lighter minor), rather than ad-hoc rules
    "axes.edgecolor": INK2, "axes.linewidth": 0.7, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.spines.top": True, "axes.spines.right": True,
    # sparingly: major gridlines only, light, with few of them. A minor grid
    # on both axes turned the panel into graph paper and competed with the data.
    "axes.grid": True, "axes.grid.which": "major", "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.5, "grid.alpha": 0.7,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False,
})

# y is the standardised residual from 05_phenotype_residualize.ipynb
YLABEL = r"$\mathrm{E}[\tilde{y}_i\,\tilde{y}_j]$"

print(f"font in use: {FONT}"
      f"   (of {FONT_STACK}, present: {[f for f in FONT_STACK if f in _have] or None})")
print(f"classes:     {CLASS_COL}")

## Load the bins

`curve()` returns bin midpoints, pair-weighted means, pair counts and per-bin
jackknife SEs for a class set. Summing the class sums *before* dividing is what
makes `noPO` a proper pair-weighted mean rather than a mean of means, and
recomputing the SE from the replicates is what makes its error bars right.

In [ ]:
CLASS_SETS = {"noPO": HE.NOPO, "PO": ["PO"]}   # `pooled` is not reported
dropped_bins = []          # (tag, class, n_dropped) -- reported after Figure 1


def curve(S, N, classes, nblocks=NBLOCKS, min_pairs=None, max_a=None,
          min_a=None, tag=""):
    """(mid, mean, n, se) for the bins this class set populates.

    Sums the class sums before dividing, so a class set is a pair-weighted mean
    rather than a mean of means, and recomputes the per-bin SE from the
    jackknife replicates so a class *set* gets a correct SE rather than a
    combination of per-class ones.
    """
    min_pairs = MIN_PAIRS_PER_BIN if min_pairs is None else min_pairs
    max_a = MAX_A if max_a is None else max_a
    min_a = -np.inf if min_a is None else min_a

    # the maths is HE.bin_curve -- shared with 09_estimators.ipynb's export, so
    # the two cannot disagree about a bin mean or a jackknife SE. Only the
    # display filtering is local.
    a, m, n, se = HE.bin_curve(S, N, classes, MID, nblocks)
    window = (a <= max_a) & (a >= min_a)
    keep = window & (n >= min_pairs)
    n_drop = int((window & ~keep).sum())
    if n_drop:
        dropped_bins.append((tag, "+".join(classes), n_drop))
    return a[keep], m[keep], n[keep], se[keep]


def load_tag(tag):
    return HE.load_arrays(f"{WORK_DIR}/{tag}_merged.full.tsv",
                          f"{WORK_DIR}/{tag}_merged.jk.tsv", len(MID), NBLOCKS)


tags = [os.path.basename(p)[: -len("_merged.full.tsv")]
        for p in sorted(glob.glob(f"{WORK_DIR}/*_merged.full.tsv"))
        if os.path.isfile(p.replace(".full.tsv", ".jk.tsv"))]
assert tags, f"no merged combo with both .full.tsv and .jk.tsv under {WORK_DIR}"

meta = pd.DataFrame([t.split("__")[:3] for t in tags],
                    columns=["phenotype", "transform", "covset"])
meta["tag"] = tags
PHENOTYPES = sorted(meta["phenotype"].unique())
TRANSFORMS = sorted(meta["transform"].unique())
COVSETS = sorted(meta["covset"].unique())
REF_TF = "invnorm" if "invnorm" in TRANSFORMS else TRANSFORMS[0]
REF_CS = "base_pcs" if "base_pcs" in COVSETS else COVSETS[0]

print(f"{len(tags)} combos | {len(PHENOTYPES)} phenotypes")
print(f"reference panel: {REF_TF} / {REF_CS}")
print(f"drawing a_ij <= {MAX_A}, bins with >= {MIN_PAIRS_PER_BIN} pairs")

## Drawing

Every figure is drawn by the same two helpers, so the clean and verbose
variants cannot drift apart.

`draw_curve` plots three series: `other` (the backbone, spanning the whole
range), `FS` and `PO`. FS and PO occupy the same bins, so they get different
markers and PO is nudged along x — they are not averaged together anywhere. The
solid red line is the **slope fitted to unrelated pairs only** (`a_ij < 0.02`,
through the origin), extended across the range as a prediction.

`verbose=True` adds, on a second copy saved alongside:

- the **pair count** on each related bin. Note the merged output stores pairs
  per bin, not individuals, so this is a pair count — an individual appears in
  many pairs.
- the **factor regression**: one slope for all related pairs plus an intercept
  offset per degree band (`h2_OneSlopeOffsets`), which is the line with bumps.
- the **full-sibs-only regression** (slope and intercept fitted to the FS class
  over 0.4–0.6).

`draw_coefs` puts the fitted coefficients with their delete-block jackknife SEs
in a narrow panel to the right of the main axes, so the numbers sit beside the
picture they come from.

In [ ]:
def fit_line(a, m, n, design):
    beta = HE._wls(design(a), m, n) if len(a) > design(a).shape[1] else None
    return beta


def draw_curve(ax, S, N, tag, verbose=False, max_a=None, min_a=None):
    """The curve itself. Returns h2_Unrel.

    Points sit above every line (zorder) and the fitted lines are thin ink, so
    a bin mean stays legible where a line runs through it without needing a
    ring around the marker.
    """
    max_a = MAX_A if max_a is None else max_a
    series = [("other", 0.0), ("FS", 0.0), ("PO", PO_XOFF)]

    ax.grid(axis="y", lw=0.6, color=GRID, zorder=0)
    ax.set_axisbelow(True)
    ax.axhline(0, color=AXIS, lw=0.7, zorder=1)

    empty = []
    for cls, xoff in series:
        col, mark = CLASS_COL[cls], CLASS_MARK[cls]
        a, m, n, se = curve(S, N, [cls], max_a=max_a, tag=tag)
        if not len(a):
            # had pairs, but no bin cleared MIN_PAIRS_PER_BIN -- say so on the
            # panel rather than leaving a silently absent series
            if N[0, HE.CLS.index(cls)].sum() > 0:
                empty.append(cls)
            continue
        # error bars behind, markers in front with a surface ring
        ax.errorbar(a + xoff, m, yerr=se, fmt="none", ecolor=col, elinewidth=0.5,
                    alpha=0.28, zorder=6)
        ax.plot(a + xoff, m, mark, ms=5.0, color=col, mec="none",
                ls="none", zorder=8,
                label=f"{CLASS_LABEL[cls]}  ({int(n.sum()):,} pairs)")
        if verbose and cls in ("FS", "PO"):
            # only the sparse first-degree bins: labelling the ~70 fine bins
            # would be unreadable, and their counts are in the millions anyway
            for x, y, cnt in zip(a + xoff, m, n):
                ax.annotate(f"{int(cnt):,}", (x, y), fontsize=5.8, color=MUTED,
                            ha="center", va="bottom", xytext=(0, 6),
                            textcoords="offset points")

    # slope from unrelated pairs only, extended as a prediction
    au, mu, nu, _ = curve(S, N, HE.NOPO, min_pairs=1, max_a=U_HI, tag=tag)
    beta_u = fit_line(au, mu, nu, lambda x: x[:, None])
    h2u = float(beta_u[0]) if beta_u is not None else np.nan
    if np.isfinite(h2u):
        x = np.linspace(0, max_a, 100)
        ax.plot(x, h2u * x, color=INK, lw=1.6, zorder=4, alpha=0.9,
                label=f"slope from unrelated pairs = {h2u:.3f}")

    if verbose:
        # all four ladder rungs, in ink, told apart by dash and a direct label
        a, m, n, _ = curve(S, N, HE.NOPO, max_a=max_a, tag=tag)
        for name, style in FIT_STYLE.items():
            ranges, design = HE.MODELS[name]
            inside = np.zeros(len(a), dtype=bool)
            for lo, hi in ranges:
                inside |= (a >= lo) & (a < hi)
            beta = fit_line(a[inside], m[inside], n[inside], design)
            if beta is None:
                continue
            xs = np.linspace(0, max_a, 400)
            pred = design(xs) @ beta
            ax.plot(xs, pred, lw=1.1, alpha=0.85, zorder=3, label=style["label"],
                    color=style["color"], ls=style["ls"])

        # full sibs only: slope and intercept fitted inside the FS range
        af, mf, nf, _ = curve(S, N, ["FS"], max_a=max_a, tag=tag)
        beta_fs = fit_line(af, mf, nf, lambda x: np.c_[x, np.ones_like(x)])
        if beta_fs is not None:
            xs = np.linspace(*FS_RANGE, 20)
            ax.plot(xs, beta_fs[0] * xs + beta_fs[1], color=CLASS_COL["FS"],
                    lw=1.8, ls=(0, (3, 1.2)), zorder=5, label="full sibs only")

    if empty:
        ax.annotate(f"{', '.join(empty)}: no bin with >= {MIN_PAIRS_PER_BIN} pairs",
                    (0.99, 0.02), xycoords="axes fraction", ha="right", va="bottom",
                    fontsize=6.5, color=MUTED)

    lo = -0.06 if min_a is None else min_a - 0.004
    ax.set_xlim(lo, max_a + 0.03)
    ax.set_xlabel("$a_{ij}$")
    ax.set_ylabel(YLABEL)
    return h2u


# Slopes and intercepts go in separate panels: a slope is a heritability on
# roughly 0-1, an offset is a shift in cross-product units, and forcing them
# onto one axis flattens whichever is smaller.
# PO-only rows are left out: with a few hundred PO pairs their SEs are wide
# enough to set the axis scale on their own, squashing everything else. The PO
# contrast is in 09_estimators.ipynb's table, where a wide interval costs
# nothing.
# Every band term of both offset models is reported, slopes and intercepts
# alike: the band intercepts come out of models that also fit slopes, so
# showing one without the other would misrepresent what was fitted.
#   "1 slope"  -> h2_OneSlopeOffsets: one slope, an intercept per band
#   "per band" -> h2_RelOffsets: a slope AND an intercept per band
SLOPE_ROWS = (
    [("h2_Unrel", "noPO", "unrelated slope"),
     ("h2_Rel", "noPO", "related slope"),
     ("h2_OneSlopeOffsets", "noPO", "1 slope: slope")]
    + [(f"RelOffsets.slope.{d}", "noPO", f"per band: slope {d}") for d in HE.OFFSET_BANDS]
    + [("b2_FS.slope", "noPO", "FS-range slope")]
)
# Per-band rows come from HE.OFFSET_BANDS, so re-cutting the bands needs no
# edit here. They are the offset terms of the model, but they ARE intercepts --
# one per relatedness band -- so they are labelled that way.
INTERCEPT_ROWS = (
    [("Unrel.intercept", "noPO", "unrelated intercept"),
     ("Rel.intercept", "noPO", "related intercept")]
    + [(f"OneSlopeOffsets.int.{d}", "noPO", f"1 slope: int {d}") for d in HE.OFFSET_BANDS]
    + [(f"RelOffsets.int.{d}", "noPO", f"per band: int {d}") for d in HE.OFFSET_BANDS]
)
# b2 is twice an intercept and lives on a different scale from the intercepts
# themselves, so it gets its own panel rather than setting their axis.
B2_ROWS = [("b2_FS", "noPO", "b2_FS  (2x FS intercept)"),
           ("b2_step", "noPO", "b2_step")]


def _coef_panel(ax, jks, rows, title, unit_range=False):
    """One row per coefficient, one marker per residualisation model."""
    tabs = {cs: jk.set_index(["estimator", "classes"]) for cs, jk in jks.items()}
    n_cs = len(tabs)
    ys, labels, lo, hi = [], [], [], []
    for est, cls, label in rows:
        if not any((est, cls) in t.index for t in tabs.values()):
            continue
        y = -len(ys)
        for k, (cs, t) in enumerate(tabs.items()):
            if (est, cls) not in t.index:
                continue
            v, se = float(t.loc[(est, cls), "est"]), float(t.loc[(est, cls), "se"])
            dy = (k - (n_cs - 1) / 2) * 0.17
            col = RAMP[k % len(RAMP)]
            hollow = np.isfinite(se) and abs(v) < se   # within 1 SE of zero
            ax.errorbar(v, y + dy, xerr=se if np.isfinite(se) else None, fmt="o",
                        ms=3.8, lw=0.7, ecolor=col, elinewidth=0.7, alpha=0.75,
                        color=col, mfc=SURFACE if hollow else col, zorder=3)
            if np.isfinite(v):
                e = se if np.isfinite(se) else 0.0
                lo.append(v - e); hi.append(v + e)
        ys.append(y); labels.append(label)

    ax.xaxis.set_major_locator(plt.MaxNLocator(6))
    # rows are categorical: no y grid, and no minor x grid either -- the
    # continuum only runs along x, and the minor lines crowded the estimates
    ax.grid(axis="y", visible=False)      # rows are categorical
    ax.axvline(0, color=INK2, lw=0.8, zorder=2)
    if unit_range:
        # heritability slopes live on 0-1; always show that range for scale,
        # widened if an estimate falls outside it
        ax.axvline(1, color=AXIS, lw=0.7, ls=":")
        pad = 0.05
        ax.set_xlim(min([0.0] + lo) - pad, max([1.0] + hi) + pad)
    ax.set_yticks(ys)
    ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_ylim(min(ys) - 0.6 if ys else -1, 0.6)
    ax.set_title(title, fontsize=9)
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)


def draw_coefs(ax_slopes, ax_ints, jks, ax_b2=None):
    """Slopes and intercepts, each on its own scale, ±1 jackknife SE.

    `jks` maps covariate set -> HE.jackknife table, so every residualisation
    model appears on the same rows in the ramp order.
    """
    _coef_panel(ax_slopes, jks, SLOPE_ROWS, "slopes  [noPO]", unit_range=True)
    _coef_panel(ax_ints, jks, INTERCEPT_ROWS, "intercepts  [noPO]")
    if ax_b2 is not None:
        _coef_panel(ax_b2, jks, B2_ROWS, "$b^2$  [noPO]")
        ax_b2.set_xlabel("coefficient ±1 SE", fontsize=8)
    else:
        ax_ints.set_xlabel("coefficient ±1 SE", fontsize=8)


print("drawing helpers defined")

# ── offset bands, from he_estimators ─────────────────────────────────────────
# HE.OFFSET_BANDS tiles [BAND_LO, R_HI): band1 0.09-0.18, band2 0.18-0.36,
# band3 0.36-0.70. The edges are KING kinship cutoffs doubled and rounded to
# bin edges, but the names make no pedigree claim -- a band is a range of a_ij.
# Both offset models are fitted on the banded range only, with no unrelated
# bins, so each is a regression on the related pairs in its own right.


# The per-degree vertical rules are gone: with a standard grid on both axes
# they were redundant clutter, and the break positions live in the model names
# and in the notebook text. HE.OFFSET_BANDS remains what the offsets are fitted on.


def draw_multi(ax, entries, verbose=False, max_a=None, min_a=None, legend=True,
               err_lw=0.5, err_alpha=0.28):
    """Every residualisation model on one axes.

    Colour carries the covariate set (an ORDERED staircase, so a sequential
    ramp rather than categorical hues); marker shape carries the pair class.
    One channel per job, so neither encoding has to be read off the other.
    """
    max_a = MAX_A if max_a is None else max_a
    ax.xaxis.set_major_locator(plt.MaxNLocator(7))
    ax.yaxis.set_major_locator(plt.MaxNLocator(5))
    ax.axhline(0, color=INK2, lw=0.6, alpha=0.5, zorder=2)

    seen = []
    for cs, S, N, col in entries:
        for cls, xoff in (("other", 0.0), ("FS", 0.0), ("PO", PO_XOFF)):
            a, m, n, se = curve(S, N, [cls], max_a=max_a, min_a=min_a, tag=cs)
            if not len(a):
                continue
            seen.append(np.c_[m - np.nan_to_num(se), m + np.nan_to_num(se)])
            # deliberately faint: the interval is context for the point, not
            # the thing being read
            ax.errorbar(a + xoff, m, yerr=se, fmt="none", ecolor=col,
                        elinewidth=err_lw, alpha=err_alpha, zorder=6)
            ax.plot(a + xoff, m, CLASS_MARK[cls], ms=4.2, color=col, mec="none",
                    ls="none", zorder=8)

    # fits come from the reference covariate set only -- one set of lines, or
    # the panel becomes unreadable
    cs_ref, S_ref, N_ref, _ = entries[0]
    a, m, n, _ = curve(S_ref, N_ref, HE.NOPO, max_a=max_a, min_a=min_a, tag=cs_ref)
    au, mu, nu, _ = curve(S_ref, N_ref, HE.NOPO, min_pairs=1, max_a=U_HI, tag=cs_ref)
    beta_u = fit_line(au, mu, nu, lambda x: x[:, None])
    h2u = float(beta_u[0]) if beta_u is not None else np.nan
    if np.isfinite(h2u):
        x = np.linspace(0, max_a, 100)
        ax.plot(x, h2u * x, color=INK, lw=1.6, zorder=4,
                label=f"unrelated-pair slope = {h2u:.3f}  [{cs_ref}]")

    if verbose:
        for name, style in FIT_STYLE.items():
            ranges, design = HE.MODELS[name]
            inside = np.zeros(len(a), dtype=bool)
            for lo, hi in ranges:
                inside |= (a >= lo) & (a < hi)
            beta = fit_line(a[inside], m[inside], n[inside], design)
            if beta is None:
                continue
            xs = np.linspace(min(0.0, a[inside].min()), a[inside].max(), 400)
            ax.plot(xs, design(xs) @ beta, lw=1.1, alpha=0.85, zorder=3,
                    color=style["color"], ls=style["ls"], label=style["label"])
        # Fitted on noPO, not on the FS class alone, so this line is exactly
        # b2_FS.slope / b2_FS from he_estimators -- the same quantity the
        # coefficient panel and the ggplot version draw. Fitting FS-only here
        # would have drawn a line no reported coefficient corresponds to.
        af, mf, nf, _ = curve(S_ref, N_ref, HE.NOPO, min_pairs=1,
                              max_a=FS_RANGE[1], tag=cs_ref)
        keep = af >= FS_RANGE[0]
        beta_fs = fit_line(af[keep], mf[keep], nf[keep],
                           lambda x: np.c_[x, np.ones_like(x)])
        if beta_fs is not None:
            xs = np.linspace(*FS_RANGE, 20)
            ax.plot(xs, beta_fs[0] * xs + beta_fs[1], color=CLASS_COL["FS"],
                    lw=1.8, ls=(0, (3, 1.2)), zorder=5, label="FS range (noPO)")

    lo = -0.06 if min_a is None else min_a - 0.004
    ax.set_xlim(lo, max_a + 0.03)
    if seen:
        allv = np.concatenate(seen).ravel()
        allv = allv[np.isfinite(allv)]
        span = allv.max() - allv.min() or 1.0
        ax.set_ylim(allv.min() - 0.08 * span, allv.max() + 0.30 * span)
    ax.set_xlabel("$a_{ij}$")
    ax.set_ylabel(YLABEL)

    if legend:
        from matplotlib.lines import Line2D
        cs_keys = [Line2D([], [], color=c, lw=2.4) for _, _, _, c in entries]
        cls_keys = [Line2D([], [], color=MUTED, marker=CLASS_MARK[k], ls="none", ms=5)
                    for k in ("other", "FS", "PO")]
        # all three legends along the top, where the data is sparse; the
        # bottom-right is left clear for the zoom inset
        l1 = ax.legend(cs_keys, [e[0] for e in entries], title="residualisation model",
                       fontsize=7, title_fontsize=7.5, loc="upper left",
                       bbox_to_anchor=(0.0, 1.0))
        ax.add_artist(l1)
        l2 = ax.legend(cls_keys, [CLASS_LABEL[k] for k in ("other", "FS", "PO")],
                       title="pair class", fontsize=7, title_fontsize=7.5,
                       loc="upper left", bbox_to_anchor=(0.0, 0.70))
        ax.add_artist(l2)
        # Fitted models go bottom-right, out of the data's way. Colour is not
        # available to tell them apart -- it is spoken for by pair class, and a
        # fourth hue on these axes fails the colourblind floors -- so they are
        # distinguished by dash pattern, and named by row in the coefficient
        # panels beside the plot.
        ax.legend(title="regression model", fontsize=7, title_fontsize=7.5,
                  loc="lower right", handlelength=3.0, labelspacing=0.3)
    return h2u


def draw_counts(ax, entries, max_a=None):
    """Pairs per bin, per class, on a log axis.

    Replaces the per-point count labels the verbose figure used to carry: at
    slide size they collided with each other and with the markers. A panel
    shows the same information as a distribution, which is what the counts are
    actually read for -- where the estimates are thin.

    NB these are PAIRS, which is what the merged output stores. An individual
    appears in many pairs, so there is no per-bin individual count available.
    """
    max_a = MAX_A if max_a is None else max_a
    cs_ref, S_ref, N_ref, _ = entries[0]
    for cls in ("other", "FS", "PO"):
        a, _, n, _ = curve(S_ref, N_ref, [cls], max_a=max_a, tag=cs_ref)
        if not len(a):
            continue
        ax.step(a, n, where="mid", lw=0.9, color=CLASS_COL[cls],
                label=f"{CLASS_LABEL[cls]}: {int(n.sum()):,}")
    ax.axhline(MIN_PAIRS_PER_BIN, color=INK2, lw=0.6, ls=(0, (3, 2)))
    ax.annotate(f"cut: {MIN_PAIRS_PER_BIN} pairs", (0.995, MIN_PAIRS_PER_BIN),
                xycoords=("axes fraction", "data"), ha="right", va="bottom",
                fontsize=6.2, color=INK2)
    ax.set_yscale("log")
    ax.set_xlim(-0.06, max_a + 0.03)
    ax.set_xlabel("$a_{ij}$")
    ax.set_ylabel("pairs per bin")
    ax.set_title("bin occupancy", fontsize=9, loc="left")
    ax.legend(fontsize=6.8, loc="upper right", ncol=1)


def draw_zoom(ax, entries, verbose=False, max_a=None):
    """The unrelated region, as its own panel under the main axes.

    Nearly every pair lives below 0.1, so this is where the unrelated slope is
    actually determined; at full scale those bins are a single blob.
    """
    zx = ZOOM_MAX if max_a is None else max_a
    # The zoom's y range is far tighter than the full panel's, so identical
    # whiskers read as much heavier there. Lighter again, for this panel only.
    draw_multi(ax, entries, verbose=verbose, max_a=zx, min_a=ZOOM_MIN, legend=False,
               err_lw=0.4, err_alpha=0.15)
    lo, hi = ax.get_ylim()          # data-driven, set inside draw_multi
    ax.set_ylim(max(lo, ZOOM_YFLOOR), hi)
    ax.set_title(f"zoom: {ZOOM_MIN} $\\leq a_{{ij}} \\leq$ {zx}", fontsize=9, loc="left")


## Figure 1 — every phenotype

One panel per phenotype at the reference transform and covariate set, drawn
twice: a clean version, and a verbose version saved alongside with pair counts,
the factor regression and the full-sibs-only fit. Duplicate/MZ pairs
(`a_ij ~ 1`) are not shown, and bins below `MIN_PAIRS_PER_BIN` pairs are
dropped.

In [ ]:
def figure_all(verbose):
    ncol = min(4, len(PHENOTYPES))
    nrow = int(np.ceil(len(PHENOTYPES) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 2.6 * nrow),
                             squeeze=False, constrained_layout=True)
    handles, labels = None, None
    for idx, phe in enumerate(PHENOTYPES):
        ax = axes[idx // ncol][idx % ncol]
        row = meta[(meta["phenotype"] == phe) & (meta["transform"] == REF_TF)
                   & (meta["covset"] == REF_CS)]
        if row.empty:
            ax.set_visible(False)
            continue
        tag = row["tag"].iloc[0]
        S, N = load_tag(tag)
        draw_curve(ax, S, N, tag, verbose=verbose)
        ax.set_title(phe, fontsize=9)
        if handles is None:
            handles, labels = ax.get_legend_handles_labels()
    for j in range(len(PHENOTYPES), nrow * ncol):
        axes[j // ncol][j % ncol].set_visible(False)

    # One legend for the whole grid, below it. Inside the first panel it sat on
    # the data -- these panels are small and the points reach the top-left
    # corner, where a legend wants to be.
    if handles:
        # "outside lower center" makes constrained_layout RESERVE the strip --
        # an ordinary lower-center legend just overlays, and landed on the
        # x-axis labels
        try:
            fig.legend(handles, labels, loc="outside lower center",
                       ncol=min(len(handles), 4), fontsize=7.5,
                       handlelength=2.6, columnspacing=1.6)
        except ValueError:      # matplotlib < 3.6 has no "outside" locations
            fig.legend(handles, labels, loc="lower center",
                       ncol=min(len(handles), 4), fontsize=7.5,
                       bbox_to_anchor=(0.5, -0.08), handlelength=2.6,
                       columnspacing=1.6)

    fig.suptitle(f"Cross-product against relatedness — {RUN_DIR}, "
                 f"{REF_TF}, {REF_CS}", fontsize=11)
    kind = "verbose" if verbose else "clean"
    png = f"{OUT_DIR}/curves_all_phenotypes_{REF_TF}_{REF_CS}_{kind}.png"
    fig.savefig(png, dpi=220, bbox_inches="tight")
    return fig, png


for verbose in (False, True):
    fig, png = figure_all(verbose)
    plt.show() if not verbose else plt.close(fig)
    print(f"-> {png}")

if dropped_bins:
    d = pd.DataFrame(dropped_bins, columns=["tag", "classes", "n_bins_dropped"])
    print(f"\nbins dropped for having < {MIN_PAIRS_PER_BIN} pairs:")
    print(d.groupby("classes")["n_bins_dropped"].max().to_string())
    print("  (max over combos; raise MIN_PAIRS_PER_BIN if this is discarding signal)")

## Figure 2 — one phenotype, with the coefficients beside it

The main axes, plus a narrow panel on the right holding every fitted slope and
intercept with its delete-block jackknife SE. Hollow markers are estimates
within one SE of zero. Saved clean and verbose.

`unrelated slope` is fitted below 0.02 only; `related slope (factor)` is the
single slope the factor model gives all related pairs; the `offset degK` rows
are that model's bumps; the `FS-range` and `PO-range` rows are slope and
intercept fitted within 0.4–0.6 on each class separately.

In [ ]:
DETAIL_PHENOTYPE = PHENOTYPES[0]   # <-- change to focus on another phenotype

# every residualisation model for this phenotype, in staircase order
_order = [c for c in ("base", "base_pcs", "base_pcs_zip3", "base_pcs_zip3_ses")
          if c in COVSETS] or COVSETS
entries, jks = [], {}
for k, cs in enumerate(_order):
    row = meta[(meta["phenotype"] == DETAIL_PHENOTYPE)
               & (meta["transform"] == REF_TF) & (meta["covset"] == cs)]
    if row.empty:
        continue
    tag = row["tag"].iloc[0]
    S, N = load_tag(tag)
    entries.append((cs, S, N, RAMP[k % len(RAMP)]))
    jks[cs] = HE.jackknife(S, N, MID, NBLOCKS)
assert entries, f"no combo for {DETAIL_PHENOTYPE} / {REF_TF}"
print(f"{DETAIL_PHENOTYPE} ({REF_TF}) — {len(entries)} residualisation models: "
      f"{[e[0] for e in entries]}")


def figure_detail(verbose):
    # sized to drop onto a 16:9 slide without being shrunk to illegibility
    nrow = 4 if verbose else 3
    fig = plt.figure(figsize=(11.0, 7.6 if verbose else 7.0),
                     constrained_layout=True)
    if verbose:
        gs = fig.add_gridspec(4, 2, width_ratios=[2.6, 1.35],
                              height_ratios=[1.2, 1.2, 0.95, 0.75])
        ax = fig.add_subplot(gs[0:2, 0])   # full range
        ax_z = fig.add_subplot(gs[2, 0])   # zoom
        ax_c = fig.add_subplot(gs[3, 0])   # bin occupancy
        ax_s = fig.add_subplot(gs[0, 1])   # slopes
        ax_i = fig.add_subplot(gs[1:3, 1]) # intercepts
        ax_b = fig.add_subplot(gs[3, 1])   # b2
    else:
        gs = fig.add_gridspec(3, 2, width_ratios=[2.6, 1.35],
                              height_ratios=[1.3, 1.25, 0.55])
        ax = fig.add_subplot(gs[0:2, 0])
        ax_z = fig.add_subplot(gs[2, 0])
        ax_c = None
        ax_s = fig.add_subplot(gs[0, 1])
        ax_i = fig.add_subplot(gs[1, 1])
        ax_b = fig.add_subplot(gs[2, 1])

    draw_multi(ax, entries, verbose=verbose)
    ax.set_title(f"{DETAIL_PHENOTYPE} — {REF_TF}, all residualisation models")
    draw_zoom(ax_z, entries, verbose=verbose)
    if ax_c is not None:
        draw_counts(ax_c, entries)
    draw_coefs(ax_s, ax_i, jks, ax_b2=ax_b)

    kind = "verbose" if verbose else "clean"
    png = f"{OUT_DIR}/curve_detail_{DETAIL_PHENOTYPE}_{REF_TF}_allcovsets_{kind}.png"
    fig.savefig(png, dpi=220, bbox_inches="tight")
    return fig, png


for verbose in (False, True):
    fig, png = figure_detail(verbose)
    plt.show() if not verbose else plt.close(fig)
    print(f"-> {png}")

## Figure 3 — zoomed on the unrelated region

`a_ij <= 0.1` holds nearly every pair, so this is where the unrelated slope is
actually determined. Error bars are per-bin jackknife SEs, so the question is
whether each bin *mean* sits on the line; with this many pairs per bin the bars
are small enough that visible departures are real. The y-axis is scaled to the
data in range — without that the panel inherits the full-range scale and shows
nothing.

In [ ]:
fig, axes = plt.subplots(1, len(TRANSFORMS), figsize=(4.8 * len(TRANSFORMS), 3.6),
                         squeeze=False, constrained_layout=True)
for ti, tf in enumerate(TRANSFORMS):
    ax = axes[0][ti]
    for ci, cs in enumerate(COVSETS):
        row = meta[(meta["phenotype"] == DETAIL_PHENOTYPE)
                   & (meta["transform"] == tf) & (meta["covset"] == cs)]
        if row.empty:
            continue
        S, N = load_tag(row["tag"].iloc[0])
        a, m, n, se = curve(S, N, HE.NOPO, max_a=ZOOM_MAX, tag=row["tag"].iloc[0])
        if not len(a):
            continue
        ax.errorbar(a, m, yerr=se, fmt="o", ms=3.2, lw=0.8, capsize=1.5,
                    color=RAMP[ci % len(RAMP)], alpha=0.9, label=cs)
        au, mu, nu, _ = curve(S, N, HE.NOPO, min_pairs=1, max_a=U_HI,
                              tag=row["tag"].iloc[0])
        beta = fit_line(au, mu, nu, lambda x: x[:, None])
        if beta is not None:
            xs = np.array([a.min(), a.max()])
            ax.plot(xs, beta[0] * xs, color=RAMP[ci % len(RAMP)], lw=1.1, ls="--",
                    alpha=0.8)
    ax.axvline(U_HI, color=MUTED, lw=0.7, ls=":")
    ax.axhline(0, color=AXIS, lw=0.6)
    ax.set_xlabel("$a_{ij}$"); ax.set_ylabel(YLABEL)
    ax.set_title(f"{DETAIL_PHENOTYPE} ({tf}) — $a_{{ij}} \\leq$ {ZOOM_MAX}\n"
                 f"dashed = slope fitted below {U_HI}", fontsize=9)
    ax.legend(fontsize=7.5, title="covariate set", title_fontsize=8)

png = f"{OUT_DIR}/curve_zoom_{DETAIL_PHENOTYPE}.png"
fig.savefig(png, dpi=220, bbox_inches="tight")
plt.show()
print(f"-> {png}")

## Export

The tables the ggplot version reads (`curves_<run>.tsv`,
`curve_params_<run>.tsv`) are written by **`09_estimators.ipynb`**, not here —
so `11r_crossproduct_curves.ipynb` depends only on the estimators notebook and
does not need this one to have run.

Both notebooks compute their bin means and jackknife SEs through
`he_estimators.bin_curve`, so the figures cannot disagree about the numbers.

## Copy notebook to bucket

In [ ]:
_nb = os.path.join(REPO_DIR, "notebooks/11_crossproduct_curves.ipynb")
_gs = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
       f"{PROJECT_DIR}/{RUN_DIR}/notebooks/11_crossproduct_curves.ipynb")
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")